# Rayleigh-Taylor Instability (2D)

This notebook runs the Rayleigh-Taylor instability benchmark: heavy fluid (`rho_high`) over light (`rho_low`) in a tall box under constant gravity `g`, perturbed by a small interface displacement `delta`. `rayleighTaylorCase.configureScheme` builds the box itself -- `aspect` times narrower than it is high, with `band` particle layers of Dirichlet buffer added above and below -- rather than using `buildDomainDescription`'s symmetric cube, and `sampleRayleighTaylor` installs both the Dirichlet boundary bands and the gravity forcing, so `buildSystem` needs nothing else.

Like `08`–`12`, this is a `particlePlot` (2D field view) case: plotting calls `buildFieldPlotter`/`refreshFieldPlotter` directly on `RAYLEIGH_TAYLOR_FIELDS` (exported from `warpSPH.cases.rayleighTaylor`) rather than going through `rayleighTaylorCase.setupPlot`/`updatePlot`, which do not live-update reliably inside a Jupyter cell in this environment. Both panels fix `vMin`/`vMax` (density around the `rho_low`/`rho_high` straddle, velocity at the buoyancy-scale `1/sqrt(2)`) so the colour scale stays comparable frame to frame instead of auto-ranging to whatever the instability has grown to by that step.

`rayleighTaylorCase` has no `timestep` hook and leaves `dt` unset in its defaults, so `sampleRayleighTaylor` picks the CFL-derived `dt` during IC construction and the loop below is a fixed `range(nSteps)`.

Precision note: switching between single and double precision is controlled in the import/configuration cell below. Because precision is set when core modules/kernels are initialized, any precision change requires a kernel restart before re-running the notebook.

![](outputs/13-Rayleigh_Taylor.gif)


In [ ]:
%matplotlib inline
from warpSPHBootstrap import bootstrap
rt = bootstrap(precision='float32', verbose=True)

from warpSPH import *
from warpSPH.cases.rayleighTaylor import rayleighTaylorCase, RAYLEIGH_TAYLOR_FIELDS
from warpSPH.cases.plotting import buildFieldPlotter, refreshFieldPlotter
from warpSPH.runner import CaseSpec, buildContext, encodeFrames
from warpSPH.io import createOutFile, prepExport, writeInitialData, writeFrame

import os
import torch
from tqdm.autonotebook import tqdm


In [ ]:
# Every knob you'd otherwise reach for as a `--flag` on
# `13-rayleigh-taylor.py`, made explicit and editable here.
# `rayleighTaylorCase.defaults`/`.params` are the same values the CLI script
# starts from -- anything not overridden below just keeps its case default.
spec = CaseSpec(caseName=rayleighTaylorCase.name, scheme=rayleighTaylorCase.scheme,
                params=dict(rayleighTaylorCase.params)) \
    .merged(**rayleighTaylorCase.defaults)

spec = spec.merged(
    # --- discretisation ------------------------------------------------
    nx=128,
    dim=2,
    L=1.0,

    # --- time stepping ---------------------------------------------------
    tLimit=10.0,
    # No `dt` here -- `sampleRayleighTaylor` leaves the CFL-derived value from
    # `computeTimestep` in `config.dt`, and there is no `timestep` hook to
    # re-pick it, so it stays fixed for the whole run.

    # --- output --------------------------------------------------------------
    plot=True, show=True, plotInterval=10,
    store=False,

    # --- Rayleigh-Taylor's own knobs (layers, gravity, box shape) -------------
    params=dict(
        gamma=1.4,
        rho_low=1.0, rho_high=2.0, delta=0.0025, g=0.5,
        aspect=2.0, band=20,
    ),
)
spec


In [ ]:
# Initial-condition generation: explicit, using the real case code
# (`rayleighTaylorCase.buildSystem` -> `sampleRayleighTaylor`), not
# re-derived here. `configureScheme` is the case's own -- it builds the tall,
# buffered box before deferring to the shared compressible setup, see the
# intro cell.
ctx = buildContext(rayleighTaylorCase, spec)
rayleighTaylorCase.configureScheme(ctx)
system = rayleighTaylorCase.buildSystem(ctx)
runningState = system.initializeNewState()


In [ ]:
# Export/plot setup via the same generic hooks `warpSPH.runner.run()` uses
# internally -- nothing here is re-derived, only called explicitly.
ctx.exportPath = prepExport(spec.caseName, ctx.config, ctx.schemeConfig, ctx.scheme, ctx.exportFunction)
spec.save(os.path.join(ctx.exportPath, 'caseSpec.json'))
print(f'exporting to {ctx.exportPath}')

# Direct buildFieldPlotter(RAYLEIGH_TAYLOR_FIELDS), not
# rayleighTaylorCase.setupPlot -- see the intro cell for why.
plotter = None
if spec.plot:
    ctx.imagePath = os.path.join(ctx.exportPath, 'images')
    os.makedirs(ctx.imagePath, exist_ok=True)
    plotter = buildFieldPlotter(ctx, runningState, RAYLEIGH_TAYLOR_FIELDS, figsize=(12, 12))

outFile = None
groups = None
if spec.store and spec.storeMode == 'trajectory':
    extraData = rayleighTaylorCase.extraData(ctx, runningState)
    outFile = createOutFile(ctx.exportPath)
    groups = writeInitialData(ctx.exportPath, outFile, ctx.scheme, ctx.config, ctx.schemeConfig,
                              spec, runningState, extraData=extraData, extraFields=rayleighTaylorCase.extraFields)


In [ ]:
# The step loop, visible and editable. This is the same call
# `warpSPH.runner.runner._run` makes internally, unrolled here so a
# perturbation or an extra diagnostic can be injected directly around it.
dt = ctx.config.dt if isinstance(ctx.config.dt, float) else ctx.config.dt.cpu().item()
nSteps = int(spec.tLimit / dt)
storeSteps = max(1, int(spec.exportInterval / dt)) if spec.storeMode == 'trajectory' \
    else max(1, spec.storeInterval)

trajectory = []
for i in (tq := tqdm(range(nSteps), leave=True)):
    # <-- hook point ---------------------------------------------------------
    stepResult = ctx.integrator.function(
        state=runningState, f=ctx.stepFunction, dt=ctx.config.dt,
        config=ctx.config, schemeConfig=ctx.schemeConfig, verbose=False,
    )
    runningState = stepResult.state
    # -------------------------------------------------------------------------

    tScalar = runningState.t.item() if torch.is_tensor(runningState.t) else runningState.t
    row = rayleighTaylorCase.diagnostics(ctx, runningState)
    trajectory.append(dict(row, step=i, t=tScalar))
    tq.set_description(f"t: {tScalar:.4f}, " + ", ".join(f"{k}: {v:.4f}" for k, v in row.items()))

    if plotter is not None and (i % spec.plotInterval == 0 or i == nSteps - 1):
        refreshFieldPlotter(ctx, runningState, plotter, RAYLEIGH_TAYLOR_FIELDS, step=i)

    if outFile is not None and (i % storeSteps == 0 or i == nSteps - 1):
        writeFrame(groups, i, stepResult.state, stepResult.stages, config=ctx.config,
                  schemeConfig=ctx.schemeConfig, uniqueParticles=True, writeStages=False,
                  extraFields=rayleighTaylorCase.extraFields)


In [ ]:
if outFile is not None:
    outFile.close()

if spec.plot:
    encodeFrames(ctx.imagePath, ctx.exportPath)
